# AI-sfa 이펙트 화풍 LoRA 학습 (Kaggle T4)
나카무라(와 보충 작가) 이펙트를 **그리는 방식**을 Illustrious-XL v1.0에 LoCon으로 학습한다. 학습 자료 1,032장 + 캡션(GPT 태그 → 영어)은 Drive 공유 링크에서, 기본 모델은 Hugging Face에서 받는다.
- 오른쪽 설정: **Accelerator = GPU T4 x2**, **Internet = On** (휴대폰 인증된 계정만 켜짐)
- 오른쪽 위 **Save Version → Save & Run All (Commit)** 으로 돌리면 창을 닫아도 최대 12시간 돈다.
- 끝나면 Output의 `out/` 폴더: `nkmr_fx_ilxl-000002.safetensors` …(2바퀴마다)와 `samples/` 그림. Output 전체를 받아 Drive에 올리고 공유 링크를 알려 주면 Claude가 바퀴별로 비교한다.
- 근거·설정 설명: 저장소 docs/PC-LORA-TRAIN-GUIDE-2026-10-25.md (Kaggle판은 T4에 맞춰 fp16 + 기본 모델 fp8, 6바퀴)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2
!free -g | head -2

In [ ]:
# 1) 학습 환경: 서버에서 불러오기까지 확인한 조합을 별도 환경(/tmp/env)에 그대로 깐다(Kaggle 기본 패키지와 섞이지 않게)
!pip -q install uv
!uv venv -q /tmp/env --python 3.10
!uv pip install -q --python /tmp/env/bin/python torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!git clone -q --depth 1 --branch v0.9.1 https://github.com/kohya-ss/sd-scripts.git /tmp/sd-scripts
!cd /tmp/sd-scripts && uv pip install -q --python /tmp/env/bin/python -r requirements.txt
!uv pip install -q --python /tmp/env/bin/python lycoris-lora==3.1.1 "numpy==1.26.4"
!cd /tmp/sd-scripts && /tmp/env/bin/python -c "import torch, library, lycoris, bitsandbytes, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 2) 학습 자료(Drive 공유 링크)와 기본 모델(Hugging Face)
!curl -sSL -o /tmp/data.zip "https://drive.usercontent.google.com/download?id=1dZ_SnaYTVQr5BL1g6loT4il6ieUBHDlX&export=download&confirm=t"
!mkdir -p /tmp/data && cd /tmp/data && unzip -q -o /tmp/data.zip && ls | wc -l
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!ls -la /tmp/ilxl.safetensors

In [ ]:
# 3) 자료 설정(캡션 섞기 끔: 텍스트 인코더 출력을 미리 계산하므로)
open('/tmp/dataset.toml','w').write('''[general]
caption_extension = ".txt"
shuffle_caption = false

[[datasets]]
resolution = 1024
batch_size = 1
enable_bucket = true
min_bucket_reso = 640
max_bucket_reso = 1536
bucket_reso_steps = 64

  [[datasets.subsets]]
  image_dir = "/tmp/data"
  num_repeats = 1
''')
print(open('/tmp/dataset.toml').read())

In [ ]:
# 4) 학습 — T4(16GB, bf16 없음): 기본 모델 fp8 보관 + fp16 계산, VAE는 fp32(그림 깨짐 방지), 6바퀴, 2바퀴마다 저장
!mkdir -p /kaggle/working/out
!cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_train_network.py \
  --pretrained_model_name_or_path /tmp/ilxl.safetensors \
  --dataset_config /tmp/dataset.toml \
  --output_dir /kaggle/working/out --output_name nkmr_fx_ilxl \
  --network_module lycoris.kohya --network_dim 32 --network_alpha 8 \
  --network_args "algo=locon" "preset=full" "conv_dim=16" "conv_alpha=4" \
  --network_train_unet_only --unet_lr 4e-4 \
  --optimizer_type AdamW8bit --lr_scheduler cosine --lr_warmup_steps 100 \
  --max_train_epochs 6 --save_every_n_epochs 2 --save_model_as safetensors \
  --mixed_precision fp16 --save_precision fp16 --fp8_base --no_half_vae --sdpa --gradient_checkpointing \
  --cache_latents --cache_latents_to_disk --cache_text_encoder_outputs --cache_text_encoder_outputs_to_disk \
  --min_snr_gamma 5 --max_data_loader_n_workers 2 --seed 42
!ls -la /kaggle/working/out

In [ ]:
# 5) 확인 그림: 같은 글·같은 시드로 기본 모델만 / 바퀴별 LoRA를 그려 비교(Output samples/)
import glob, os
prompts = [
 "nkmr_fx, anime screencap, explosion, smoke, billowing mass, white core orange rim, layered cel shading, no outline, sharp tips, expanding, large effect, no humans, simple background, wide shot",
 "nkmr_fx, anime screencap, lightning, embers, streaks, particles, white core blue rim, no outline, sharp tips, rapid expansion, no humans, city, wide shot",
 "nkmr_fx, anime screencap, impact frame, flash, radial, monochrome, rough brush strokes, peak, full screen effect, no humans, abstract background",
 "nkmr_fx, anime screencap, debris, dust cloud, billowing mass, flat color, simple cel shading, dispersing, flying, no humans, rocky ground, wide shot",
 "nkmr_fx, anime screencap, energy beam, shockwave, ring, white core pink rim, layered cel shading, sustained, large effect, no humans, sky",
 "nkmr_fx, anime screencap, fire, smoke, pillar, white core orange rim, layered cel shading, rough brush strokes, expanding, anime character, city, from below",
]
open('/tmp/prompts.txt','w').write("\n".join(p + " --n lowres, worst quality, text, watermark --w 1344 --h 768 --s 28 --l 6 --d 1234" for p in prompts))
ckpts = sorted(glob.glob('/kaggle/working/out/nkmr_fx_ilxl*.safetensors'))
print(ckpts)
runs = [("base", None)] + [(os.path.basename(c)[:-12], c) for c in ckpts]
for name, c in runs:
    out = f"/kaggle/working/samples/{name}"
    os.makedirs(out, exist_ok=True)
    net = f"--network_module lycoris.kohya --network_weights {c} --network_mul 1.0 --network_merge" if c else ""  # LyCORIS 3.1.1은 apply_to가 flag not set으로 멈춘다(2026-10-25 Kaggle 1차) → 합쳐서 그림
    os.system(f"cd /tmp/sd-scripts && /tmp/env/bin/python sdxl_gen_img.py --ckpt /tmp/ilxl.safetensors --outdir {out} --from_file /tmp/prompts.txt --images_per_prompt 1 --fp16 --no_half_vae --sdpa --sampler euler_a {net} > {out}/log.txt 2>&1")
    print(name, len(glob.glob(out + '/*.png')), 'images')

In [ ]:
# 6) 확인 그림을 한 장으로(바퀴별 줄, 글별 칸) + 받기 쉽게 묶기
from PIL import Image
rows = sorted(glob.glob('/kaggle/working/samples/*/'))
grid = []
for r in rows:
    ims = sorted(glob.glob(r + '*.png'))[:6]
    if ims:
        grid.append([Image.open(p).resize((448, 256)) for p in ims])
if grid:
    W = Image.new('RGB', (448 * max(len(g) for g in grid), 256 * len(grid)), 'black')
    for y, g in enumerate(grid):
        for x, im in enumerate(g):
            W.paste(im, (448 * x, 256 * y))
    W.save('/kaggle/working/samples_grid.jpg', quality=85)
    print('rows:', [os.path.basename(r.rstrip('/')) for r in rows])
!cd /kaggle/working && zip -q -r lora_result.zip out samples samples_grid.jpg && ls -la /kaggle/working